# Stochastic Calculus from Zero · Episode 16: Two Brownian motions

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path is simulated with a fixed seed. The correlation, 0.5, is illustrative.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Building correlation · 4. Cross variation · 5. The product rule · 6. Lévy's characterisation · 7. Part 2 checkpoint · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 16,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

$f(x) = x^3$: $d(W^3) = 3W^2\,dW + 3W\,dt$.

In [ ]:
out["prev"] = {"answer": "3W^2 dW + 3W dt"}

## 3. Building correlation

Take two independent Brownian motions $W_1$ and $B$ and set

$$W_2 = \rho\,W_1 + \sqrt{1 - \rho^2}\;B .$$

$W_2$ is again a Brownian motion (its increments are normal with variance $(\rho^2 + 1 - \rho^2)\Delta t = \Delta t$), and $\mathrm{Corr}(W_1(t), W_2(t)) = \rho$. This is the two-dimensional **Cholesky** factorisation of the correlation matrix.

In [ ]:
rng = np.random.default_rng(seed)
rho = 0.5
N = 2 ** 16
t, A, B = correlated_paths(1, N, rho, rng)
a, b = A[0], B[0]
every = 64
out["pair"] = {"rho": rho, "t": t[::every].round(5).tolist(), "W1": [a[::every].round(4).tolist()], "W2": b[::every].round(4).tolist(),
               "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))],
               "y_lo": float(min(a.min(), b.min())), "y_hi": float(max(a.max(), b.max()))}

## 4. Cross variation

Multiply the two paths' steps and add up: $\sum \Delta W_1\,\Delta W_2 \to \rho\,t$. The multiplication table (Episode 11) gains a row: $dW_1 \cdot dW_2 = \rho\,dt$.

In [ ]:
cross = []
for n in (16, 256, 4_096, N):
    s = N // n
    cross.append({"steps": n, "cross": float((np.diff(a[::s]) * np.diff(b[::s])).sum())})
assert abs(cross[-1]["cross"] - rho) < 0.03
out["cross"] = {"rows": cross, "fine": cross[-1]["cross"]}
pd.DataFrame(cross)

## 5. The product rule

For any two paths, $a_T b_T - a_0 b_0 = \sum a\,\Delta b + \sum b\,\Delta a + \sum \Delta a\,\Delta b$ exactly (left-point sums: algebra again). In the limit:

$$d(W_1 W_2) = W_1\,dW_2 + W_2\,dW_1 + \rho\,dt .$$

The Itô integrals have mean zero, so $E[W_1(t)\,W_2(t)] = \rho\,t$.

In [ ]:
lhs = a[-1] * b[-1]
rhs = (a[:-1] * np.diff(b)).sum() + (b[:-1] * np.diff(a)).sum() + (np.diff(a) * np.diff(b)).sum()
_, A2, B2 = correlated_paths(20_000, 100, rho, rng)
prod = A2[:, -1] * B2[:, -1]
out["product"] = {"identity_error": float(abs(lhs - rhs)), "E_W1W2": float(prod.mean()), "se": float(prod.std() / math.sqrt(len(prod)))}
assert out["product"]["identity_error"] < 1e-9 and abs(out["product"]["E_W1W2"] - rho) < 4 * out["product"]["se"]
out["product"]

## 6. Lévy's characterisation

A continuous martingale that starts at 0 and has quadratic variation $t$ is a Brownian motion: **Lévy's characterisation**. So we can recover the independent driver from the pair: $B = (W_2 - \rho W_1)/\sqrt{1 - \rho^2}$ is a continuous martingale with $[B]_t = t$, and its cross variation with $W_1$ is zero: it is a Brownian motion independent of $W_1$.

In [ ]:
indep = (b - rho * a) / math.sqrt(1 - rho ** 2)
ends = (B2[:, -1] - rho * A2[:, -1]) / math.sqrt(1 - rho ** 2)
out["levy"] = {"qv": float((np.diff(indep) ** 2).sum()), "cross_with_W1": float((np.diff(indep) * np.diff(a)).sum()),
               "corr": float(np.corrcoef(A2[:, -1], ends)[0, 1]), "var": float(ends.var())}
assert abs(out["levy"]["qv"] - 1) < 0.02 and abs(out["levy"]["cross_with_W1"]) < 0.02 and abs(out["levy"]["var"] - 1) < 0.04
out["levy"]

## 7. Part 2 checkpoint

1. $\mathrm{Cov}(W_2, W_7)$? **2**, the earlier time (Episode 9).
2. Quadratic variation of $2W_t + 5t$ over $[0, 3]$? **12**: $2^2 \times 3$; the drift adds nothing (Episode 11).
3. $d(W^4)$, and the martingale it implies? $d(W^4) = 4W^3\,dW + 6W^2\,dt$, so $W_t^4 - 6\int_0^t W_u^2\,du$ is a martingale (Episodes 12 and 15).
4. $\mathrm{Var}\big(\int_0^1 W^2\,dW\big)$? **1**: by the isometry, $\int_0^1 E[W_t^4]\,dt = \int_0^1 3t^2\,dt$ (Episode 14).

In [ ]:
_, W4, dW4 = brownian_paths(20_000, 400, 1.0, rng)
m4 = W4[:, -1] ** 4 - 6 * (W4[:, :-1] ** 2).sum(axis=1) / 400
J = (W4[:, :-1] ** 2 * dW4).sum(axis=1)
out["checkpoint"] = {"q1": 2, "q2": 4 * 3, "q3_mean": float(m4.mean()), "q3_se": float(m4.std() / math.sqrt(len(m4))),
                     "q4": float(J.var())}
assert abs(out["checkpoint"]["q3_mean"]) < 4 * out["checkpoint"]["q3_se"] and abs(out["checkpoint"]["q4"] - 1) < 0.1
out["checkpoint"]

## 8. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** With $\rho = 0$, what is $d(W_1 W_2)$? What is $E[W_1(t) W_2(t)]$?

<details><summary>Answer</summary>

$d(W_1W_2) = W_1\,dW_2 + W_2\,dW_1$: no extra term. $E[W_1W_2] = 0$.

</details>

**Exercise 2 (By hand).** What is $\mathrm{Var}(W_1(t) + W_2(t))$ when their correlation is $\rho$?

<details><summary>Answer</summary>

$t + t + 2\rho t = 2(1 + \rho)t$.

</details>

**Exercise 3 (By hand).** What is the quadratic variation of $W_1 + W_2$ over $[0, t]$?

<details><summary>Answer</summary>

$(dW_1 + dW_2)^2 = dt + dt + 2\rho\,dt$, so $2(1 + \rho)t$: the same as the variance.

</details>

**Exercise 4 (In code).** Build a pair with $\rho = -0.8$ and check the cross variation on a fine grid.

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
_, P1, P2 = correlated_paths(1, 2 ** 16, -0.8, np.random.default_rng(seed + 16))
cv = float((np.diff(P1[0]) * np.diff(P2[0])).sum())
assert abs(cv + 0.8) < 0.03
print(f"cross variation {cv:+.3f} (rho = -0.8)")

**Exercise 5 (In code).** Build three Brownian motions with pairwise correlations 0.5, 0.3 and 0.2 from a 3 × 3 Cholesky factor, and check their sample correlations.

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
C = np.array([[1, 0.5, 0.3], [0.5, 1, 0.2], [0.3, 0.2, 1]])
L = np.linalg.cholesky(C)
Z = np.random.default_rng(seed + 160).standard_normal((3, 200_000))
X = L @ Z
est = np.corrcoef(X)
assert np.abs(est - C).max() < 0.01
print(np.round(est, 3))

## 9. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")